# Stage 5: Model Final Evaluation and Comparison

This section assumes `X_train, X_test, y_train, y_test` already exist from the data-splitting stage (80/20 stratified split, malignant = 1). The test set is used here for the first and only time.

## 1. Final models (hyperparameters selected in Stage 4)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, roc_curve)

RS = 42

# Skip this class if it is already defined in your Stage 4 code
class IQRMedianReplacer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        q1, q3 = np.percentile(X, 25, axis=0), np.percentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lo_, self.hi_ = q1 - 1.5*iqr, q3 + 1.5*iqr
        self.med_ = np.median(X, axis=0)
        return self
    def transform(self, X):
        X = np.array(X, float)
        return np.where((X < self.lo_) | (X > self.hi_), self.med_, X)

def pipe(m):
    return Pipeline([("iqr", IQRMedianReplacer()), ("scale", StandardScaler()), ("model", m)])

models = {
    "Logistic Regression": pipe(LogisticRegression(C=0.1, max_iter=1000, random_state=RS)),
    "SVM (RBF)": pipe(SVC(kernel="rbf", C=10, gamma=0.001, probability=True, random_state=RS)),
    "Random Forest": pipe(RandomForestClassifier(n_estimators=100, max_depth=None,
                          min_samples_leaf=1, max_features=0.5, random_state=RS)),
    "Neural Network": pipe(MLPClassifier(hidden_layer_sizes=(16, 8), activation="relu",
                          solver="adam", alpha=0.1, learning_rate_init=0.01, batch_size=32,
                          max_iter=300, early_stopping=True, validation_fraction=0.15,
                          n_iter_no_change=20, random_state=RS)),
}
for m in models.values():
    m.fit(X_train, y_train)

## 2. Test-set metrics (Table 5.1 and Table 5.4)

In [ ]:
rows, probs, preds = [], {}, {}
for name, m in models.items():
    pred = m.predict(X_test); p = m.predict_proba(X_test)[:, 1]
    preds[name], probs[name] = pred, p
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rows.append({"Model": name,
                 "Accuracy": accuracy_score(y_test, pred),
                 "Precision": precision_score(y_test, pred),
                 "Recall": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred),
                 "ROC-AUC": roc_auc_score(y_test, p),
                 "TN": tn, "FP": fp, "FN": fn, "TP": tp,
                 "Train Acc": accuracy_score(y_train, m.predict(X_train))})
results = pd.DataFrame(rows).set_index("Model")
display(results.round(4))

## 3. Confusion matrices (Figure 5.1)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, (name, pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, pred)
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=16,
                    color="white" if cm[i, j] > cm.max()/2 else "black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Benign", "Malignant"]); ax.set_yticklabels(["Benign", "Malignant"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(name)
plt.tight_layout(); plt.show()

## 4. ROC curves (Figure 5.2)

In [ ]:
plt.figure(figsize=(5.6, 4.8))
for name, p in probs.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc_score(y_test, p):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate (recall)")
plt.title("ROC curves on the test set"); plt.legend(loc="lower right", fontsize=8)
plt.grid(alpha=0.3); plt.show()

## 5. Final comparison and most suitable model (Figure 5.3)

In [ ]:
metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
results[metrics].plot(kind="bar", figsize=(9, 4), ylim=(0.85, 1.04))
plt.title("Test-set metrics of the four models"); plt.xticks(rotation=0)
plt.legend(ncol=5, loc="upper center", fontsize=8); plt.grid(axis="y", alpha=0.3); plt.show()

print("Best by F1:", results["F1"].idxmax())
print("Train - Test accuracy gap (pts):")
print(((results["Train Acc"] - results["Accuracy"]) * 100).round(1))